# Train Khatib v8 on ColabTrains the NNUE on Lichess's Stockfish-evaluation database.**Before you run:** Runtime -> Change runtime type -> **GPU** (L4 or T4).Free Colab can disconnect at any time, so every epoch is checkpointed toGoogle Drive. If it drops, re-run the notebook and it resumes from the lastcheckpoint rather than starting over.

In [ ]:
# 1. Check the GPU. Stop here if this says 'No GPU'.import subprocessprint(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],                     capture_output=True, text=True).stdout or 'No GPU -- set Runtime > Change runtime type > GPU')

In [ ]:
# 2. Mount Drive so checkpoints survive a disconnect.from google.colab import drivedrive.mount('/content/drive')import osCKPT = '/content/drive/MyDrive/khatib'os.makedirs(CKPT, exist_ok=True)print('checkpoints ->', CKPT)

In [ ]:
# 3. Get the trainer.!git clone -q https://github.com/Nesbesss/khatib-chess /content/khatib 2>/dev/null || (cd /content/khatib && git pull -q)!pip install -q zstandardprint('ready')

In [ ]:
# 4. Stream Lichess's eval database into training data.#    ~395M positions available; POSITIONS sets how many we keep.#    Downloads at datacenter speed -- a few minutes, not hours.POSITIONS = 95_000_000import osDATA = '/content/lichess_evals.txt'if os.path.exists(DATA) and os.path.getsize(DATA) > 1e9:    print('data already present:', round(os.path.getsize(DATA)/1e9, 1), 'GB')else:    !cd /content/khatib && python3 -u trainer/convert_lichess_evals.py \        --out {DATA} --limit {POSITIONS}print('size:', round(os.path.getsize(DATA)/1e9, 2), 'GB')print('lines:', sum(1 for _ in open(DATA)))

In [ ]:
# 5. Train. Checkpoints land in Drive every epoch.#    An L4 does roughly 20-50x the M4's rate, so this is hours, not days.#    If Colab disconnects, just re-run this cell: it resumes from state.pt.EPOCHS = 20LIMIT  = 60_000_000     # positions actually used; raise if RAM allowsimport osos.environ['CHESS_HIDDEN'] = '2048'!cd /content/khatib && python3 -u trainer/train.py \    --data {DATA} --limit {LIMIT} \    --out {CKPT}/v8.nnue \    --epochs {EPOCHS} --batch 16384 --lr 1e-3 --lambda 0.7 \    --checkpoint-every 1 --resume-state {CKPT}/state.pt

In [ ]:
# 6. Confirm the net came out at the size the engine expects.import os, globfor p in sorted(glob.glob(CKPT + '/v8.nnue*')):    print(f'{os.path.basename(p):24} {os.path.getsize(p):,} bytes')print('\nexpected 26,219,024 bytes for the 2048-wide net')print('Download from Drive, drop in as net.nnue, then test it against v7 in games.')